In [ ]:
!apt-get install -y software-properties-common
!apt-add-repository -y ppa:swi-prolog/stable
!apt-get update
!apt-get install -y swi-prolog

In [ ]:
!pip install pyswip

In [1]:
import os
from pyswip import Prolog

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
directory = "/content/drive/My Drive/KOA RecomSys/"

In [4]:
entities_path = os.path.join(directory, "entities.pl")
triples_path = os.path.join(directory, "triples.pl")
rules_path = os.path.join(directory, "rules.pl")

In [5]:
prolog = Prolog()

In [6]:
def load_knowledge_base():
    """Carrega os arquivos Prolog no motor de inferência."""
    try:
        # Consultando os arquivos (o SWI-Prolog usa caminhos com barras normais /)
        prolog.consult(rules_path.replace("\\", "/"))
        prolog.consult(entities_path.replace("\\", "/"))
        prolog.consult(triples_path.replace("\\", "/"))
        print("✅ Base de Conhecimento Ontológica carregada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao carregar arquivos Prolog: {e}")

In [8]:
def get_explainable_recommendations(user_id, limit=3):
    """
    Executa a query de unpacking ontológico e retorna recomendações explicadas.
    """
    query = f"explain_recommendation('{user_id}', ProductID, Explanation)"
    results = []

    # Executa a query e itera sobre as soluções encontradas
    for soln in prolog.query(query):
        product_id = str(soln["ProductID"])
        explanation = str(soln["Explanation"])

        if explanation not in [r['explanation'] for r in results]:
            results.append({
                "product_id": product_id,
                "explanation": explanation
            })

        if len(results) >= limit:
            break

    return results

In [10]:
def get_detailed_recommendations(user_id, limit=3):
    """
    Executa a query capturando não apenas a explicação, mas também
    os dados de origem (BoughtProd, UnpackedType, etc.)
    """
    # Alteramos a query para extrair as variáveis internas da regra de recomendação
    # Nota: Precisamos que a regra 'recommend' ou 'explain' no rules.pl exponha BoughtProd
    query = f"recommend('{user_id}', ProductID, Reason), functional_complex(ProductID, Name, _, _, _), buys('{user_id}', BoughtProdID), functional_complex(BoughtProdID, BoughtName, _, _, _), unpack_relation(BoughtProdID, ProductID, Type)"

    results = []

    # Executamos a query
    for soln in prolog.query(query):
        res = {
            "target_id": str(soln["ProductID"]),
            "target_name": soln["Name"].decode('utf-8') if isinstance(soln["Name"], bytes) else soln["Name"],
            "source_id": str(soln["BoughtProdID"]),
            "source_name": soln["BoughtName"].decode('utf-8') if isinstance(soln["BoughtName"], bytes) else soln["BoughtName"],
            "unpacked_type": soln["Type"].decode('utf-8') if isinstance(soln["Type"], bytes) else soln["Type"],
            "reason_text": soln["Reason"].decode('utf-8') if isinstance(soln["Reason"], bytes) else soln["Reason"]
        }

        # Evitar duplicatas de recomendação para o mesmo produto
        if res["target_id"] not in [r['target_id'] for r in results]:
            results.append(res)

        if len(results) >= limit:
            break

    return results

In [7]:
# 3. Execução Principal
load_knowledge_base()

✅ Base de Conhecimento Ontológica carregada com sucesso!


In [9]:
# Exemplo: Testando para um usuário específico (ex: U0010 ou um que tenha dados)
target_user = "U0534"
recommendations = get_explainable_recommendations(target_user)

if not recommendations:
    print(f"\nNenhuma recomendação encontrada para {target_user} baseada nas regras ontológicas.")
else:
    print(f"\n--- Recomendações Explicáveis para {target_user} ---")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. [ID: {rec['product_id']}]")
        print(f"   Explicação Lógica: {rec['explanation']}\n")



--- Recomendações Explicáveis para U0534 ---
1. [ID: P3110]
   Explicação Lógica: Hello Nicholas Zavala, we suggest Dell and Sons Cloud-Ready Accessories Model-400. Reason: Based on previous purchases, this provides Solution Composition.

2. [ID: P0795]
   Explicação Lógica: Hello Nicholas Zavala, we suggest Logitech and Sons Mechanical Cables Model-928. Reason: Based on previous purchases, this provides Solution Composition.

3. [ID: P0026]
   Explicação Lógica: Hello Nicholas Zavala, we suggest Dell Inc Gaming Networking Model-500. Reason: Based on previous purchases, this provides Solution Composition.



In [12]:
target_user = "U0534"
recommendations = get_detailed_recommendations(target_user)

if not recommendations:
    print(f"\nNenhuma recomendação encontrada para {target_user} baseada nas regras ontológicas.")
else:
    # Captura o nome do usuário para o cabeçalho
    user_info = list(prolog.query(f"agent('{target_user}', Name, Profile)"))
    u_name = user_info[0]['Name'] if user_info else target_user

    print(f"\n" + "="*60)
    print(f"SISTEMA DE RECOMENDAÇÃO AUTO-EXPLICÁVEL (UFO-BASED)")
    print(f"Usuário: {u_name} ({target_user})")
    print("="*60)

    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. PRODUTO SUGERIDO: {rec['target_name']} [{rec['target_id']}]")
        print(f"   |")
        print(f"   +-- DADO DE ORIGEM: O usuário já possui '{rec['source_name']}' [{rec['source_id']}]")
        print(f"   |")
        print(f"   +-- FUNDAMENTAÇÃO ONTOLÓGICA (Unpacking): {rec['unpacked_type']}")
        print(f"   |")
        print(f"   +-- JUSTIFICATIVA FINAL: {rec['reason_text']}")
        print(f"   " + "-"*40)


SISTEMA DE RECOMENDAÇÃO AUTO-EXPLICÁVEL (UFO-BASED)
Usuário: Nicholas Zavala (U0534)

1. PRODUTO SUGERIDO: Dell and Sons Cloud-Ready Accessories Model-400 [P3110]
   |
   +-- DADO DE ORIGEM: O usuário já possui 'Samsung Ltd Wireless Peripherals Model-118' [P0514]
   |
   +-- FUNDAMENTAÇÃO ONTOLÓGICA (Unpacking): Solution Composition
   |
   +-- JUSTIFICATIVA FINAL: Based on previous purchases, this provides Solution Composition.
   ----------------------------------------

2. PRODUTO SUGERIDO: Logitech and Sons Mechanical Cables Model-928 [P0795]
   |
   +-- DADO DE ORIGEM: O usuário já possui 'Samsung LLC Gaming Storage Model-735' [P0762]
   |
   +-- FUNDAMENTAÇÃO ONTOLÓGICA (Unpacking): Solution Composition
   |
   +-- JUSTIFICATIVA FINAL: Based on previous purchases, this provides Solution Composition.
   ----------------------------------------

3. PRODUTO SUGERIDO: Dell Inc Gaming Networking Model-500 [P0026]
   |
   +-- DADO DE ORIGEM: O usuário já possui 'Samsung LLC Gaming Sto

In [ ]:
# 4. Preparação para o LLM (Opcional)
def format_for_llm(user_id, recommendations):
    """Formata os dados do Prolog para um prompt de Fine-Tuning ou RAG."""
    prompt = f"O sistema de raciocínio lógico identificou as seguintes recomendações para o usuário {user_id}:\n"
    for rec in recommendations:
        prompt += f"- {rec['explanation']}\n"
    prompt += "\nPor favor, reescreva essas explicações de forma amigável e persuasiva para o usuário."
    return prompt

In [ ]:
# Exemplo de prompt que seria enviado ao GPT-4
print(format_for_llm(target_user, recommendations))